# BGE-RERANKER-BASE深度语义匹配  
* 直接利用attention机制的token级语义联系，实现深入语义匹配精召回。  
* **通常在粗召回（语义、关键词、混合）初筛后，再用reranker对结果精排。**  
* 在milvus中，reranker（严格意义上的reranker）可以用于直接向量检索与混合检索。其在混合检索中作为function其实是不太合理的：①在混合检索中使用意味着可能要跑重复数据，不符合reranker的工程使用思路（very expensive）②因其只反映语义匹配，在特殊情况下（比如参考了关键词）无法起到混合多路检索的作用（无法公正对待关键词匹配）  
* reranker可作为function传入`search`或hybrid_search（不建议后者，因此没有举例。可查看官方文档或模仿milvus_粗召回.RRF混合检索），也可以`自行调用模型`（见RRF + BGE-Reranker）。

In [2]:
from pymilvus import MilvusClient, DataType, Function, FunctionType, AnnSearchRequest

client = MilvusClient(uri="http://localhost:19530", token="root:Milvus")

# 这里分别创建BM25的spasevector、bge的densevector、支持bm25的text的schema
schema = MilvusClient.create_schema()

schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True, auto_id=True)
schema.add_field(field_name="text", datatype=DataType.VARCHAR, max_length=8194, enable_analyzer=True)
schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)

bm25_function = Function(name="bm25_func", function_type=FunctionType.BM25, input_field_names=["text"], output_field_names=["sparse"])
bge_function = Function(name="bge-m3-func", function_type=FunctionType.TEXTEMBEDDING, input_field_names=["text"], output_field_names=["dense"],
params={
    "provider": "TEI",
    "endpoint": "http://host.docker.internal:8080",
    "truncate": "true"
})

schema.add_function(bm25_function)
schema.add_function(bge_function)

# 创建索引
index_params = client.prepare_index_params()
index_params.add_index(field_name="sparse", index_name="sparse_index", index_type="SPARSE_INVERTED_INDEX", metric_type="BM25",
params={
    "inverted_index_algo": "DAAT_MAXSCORE",
    "bm25_k1": 1.2,
    "bm25_b": 0.75
})
index_params.add_index(field_name="dense", index_name="dense_index", index_type="AUTOINDEX", metric_type="COSINE")

# 创建collection
client.create_collection(
    collection_name="reranker_test",
    schema=schema,
    index_params=index_params,
    consistency_level="Strong",
)
client.load_collection(collection_name="reranker_test")

# 写入数据
data = [{"text": "Red cotton t-shirt with round neck"}, {"text": "Wireless noise-cancelling over-ear headphones"}, {"text": "Stainless steel water bottle, 500ml"}]
client.insert(collection_name="reranker_test", data=data)

{'insert_count': 3, 'ids': [463657230101652603, 463657230101652604, 463657230101652605], 'cost': 0}

## search中定义BGE-Reranker-Base的function（基于HuggingFace TEI）
* function参数包括：  
    * name：函数名称  
    * input_field_names：输入字段名称  
    * function_type：FunctionType.RERANK  
    * params：  
        * "queries": ["需要匹配的query"]
        * "reranker": "model"。此处填写固定字段model
        * "provider": "tei"。此处服务为tei
        * "endpoint": "http://host.docker.internal:8081"。此处填写TEI服务地址（区分localhost与host.docker.internal）
        * "truncate": True。可选，是否截断文本。
        * "truncation_direction": "Right"。可选，截断方向，默认Right。
        * "max_client_batch_size": 32。可选，最大客户端批次大小，默认32。
* 传入search，根据初筛结果进行精排序

In [3]:
query_text = "white headphones, quiet and comfortable"

In [4]:
bge_rerank_func = Function(
    name="bge_rerank_func",
    input_field_names=["text"],
    function_type=FunctionType.RERANK,
    params={
        "queries": [query_text],
        "reranker": "model",
        "provider": "tei",
        "endpoint": "http://host.docker.internal:8081",
        "truncate": True,
        # "truncation_direction": "Right",
        # "max_client_batch_size": 32,
    }
)

### BGE-M3 + BGE-Reranker-Base (search)

In [5]:
results = client.search(
    collection_name="reranker_test",
    data=[query_text],
    anns_field="dense",
    limit=2, # 三个里面先筛出两个，仅作示例
    output_fields=["text"],
    ranker=bge_rerank_func, # 传入reranker函数负责对bgem3选出的两个重新精排
    consistency_level="Strong"
)
print(results)

data: [[{'id': 463657230101652604, 'distance': 0.37501955032348633, 'entity': {'text': 'Wireless noise-cancelling over-ear headphones'}}, {'id': 463657230101652605, 'distance': 3.734356869244948e-05, 'entity': {'text': 'Stainless steel water bottle, 500ml'}}]]


### BM25 + BGE-Reranker-Base (search)

In [6]:
results2 = client.search(
    collection_name="reranker_test",
    data=[query_text],
    anns_field="sparse",
    limit=2,
    output_fields=["text"],
    ranker=bge_rerank_func,
    consistency_level="Strong"
)
print(results2)

data: [[{'id': 463657230101652604, 'distance': 0.37501955032348633, 'entity': {'text': 'Wireless noise-cancelling over-ear headphones'}}]]


## RRF (BM25 + BGE-M3) + BGE-Reranker-Base (自行调用)  
hybrid_search初筛后，自行调用模型精排  
自行调用模型有三种方式：  
* 继续使用Hugging Face TEI的服务端口  
* 使用milvus配置好的模型库  
* 本地pytorch调用hugging face模型

In [8]:
# 混合检索先召回结果，3筛2（仅示例）
dense_search = AnnSearchRequest(
    data=[query_text],
    anns_field="dense",
    param={},
    limit=3,
)
sparse_search = AnnSearchRequest(
    data=[query_text],
    anns_field="sparse",
    param={},
    limit=3,
)
reqs = [dense_search, sparse_search]

rrf_func = Function(
    name="rrf",
    input_field_names=[], # 必须为空
    function_type=FunctionType.RERANK,
    params={
        "reranker": "rrf",
        "k": 60
    }
)

hybrid_results = client.hybrid_search(
    collection_name="reranker_test",
    reqs=reqs,
    ranker=rrf_func,
    output_fields=["text"],
    limit=2
)

In [9]:
# 用于整合混合检索返回的数据格式
def pharse_results(hybrid_results):
    results = []
    res_dicts = hybrid_results[0]
    for res_dict in res_dicts:
        current_dict = {}
        for key, value in res_dict.items():
            if key == 'entity':
                entity = res_dict['entity']
                if entity:
                    for ent_key, ent_value in entity.items():
                        current_dict[ent_key] = ent_value
            else:
                current_dict[key] = value
        results.append(current_dict)
    return results

In [13]:
results_hybrid = pharse_results(hybrid_results)
print(results_hybrid)
documents = [item['text'] for item in results_hybrid]
print(documents)

[{'id': 463657230101652604, 'distance': 0.032786883413791656, 'text': 'Wireless noise-cancelling over-ear headphones'}, {'id': 463657230101652605, 'distance': 0.016129031777381897, 'text': 'Stainless steel water bottle, 500ml'}]
['Wireless noise-cancelling over-ear headphones', 'Stainless steel water bottle, 500ml']


### 使用TEI服务（自行调用）  
需要提供接口地址，例如: `http://localhost:8081/rerank`  
通常接口服务处，嵌入为`/embed`，reranker为`/rerank`  
对reranker来说，请求体常用参数包括`query`(str)、`texts`(List[str])、`truncate`(bool)、`return_text`(bool)

In [20]:
import requests
import json

In [21]:
# 调用TEI服务的reranker接口
reranker_url = "http://localhost:8081/rerank"
# 请求体
payload = {
    "query": query_text,
    "texts": documents,
    "truncate": True,
    "return_text": True
}

In [22]:
try:
    response = requests.post(url=reranker_url, json=payload)
    response.raise_for_status()
    results = response.json()
    print(results)
except Exception as e:
    print(f"Error: {e}")

[{'index': 0, 'text': 'Wireless noise-cancelling over-ear headphones', 'score': 0.37501955}, {'index': 1, 'text': 'Stainless steel water bottle, 500ml', 'score': 3.734357e-05}]


### 使用Milvus模型库（自行调用）  
参考`milvus_便捷模型调用.ipynb`  
milvus提供了丰富的embed（https://milvus.io/docs/zh/embeddings.md）与rerank模型（https://milvus.io/docs/zh/rerankers-overview.md）  
此处我们使用官方接口调用BGE-Reranker-Base:  
* 需要使用`BGERerankFunction(model_name, device)`初始化实例  
* 调用实例时，需要传入query文本、documents列表、top_k（可选）

In [23]:
from pymilvus.model.reranker import BGERerankFunction

In [26]:
bge_rf = BGERerankFunction(
    model_name="BAAI/bge-reranker-base",
    device="cuda:0"
)

In [28]:
results = bge_rf(
    query=query_text,
    documents=documents,
    # top_k=3,
)
print(results)

[RerankResult(text='Wireless noise-cancelling over-ear headphones', score=0.3736472447989349, index=0), RerankResult(text='Stainless steel water bottle, 500ml', score=3.734356896287336e-05, index=1)]
